# Credit Card Fraud Detection - Complete End-to-End Pipeline

This comprehensive master notebook integrates all stages of the Credit Card Fraud Detection system into a single workflow:
1. Data Exploration and Statistical Inspection
2. Target Variable and Feature Distribution Analysis
3. Correlation Analysis and Discriminating Feature Identification
4. Feature Engineering and Data Preprocessing
5. Baseline and Balanced Logistic Regression
6. Constrained Decision Tree Classification
7. Scaled K-Nearest Neighbor Classification
8. Advanced Random Forest with SMOTE Oversampling
9. Unified Model Benchmark and Comparison
10. Deep-Dive Financial Cost and Threshold Evaluation
11. In-Memory Production Pipeline and Real-Time Scoring


## 1. Imports and Global Configuration

In [ ]:
import os
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report, brier_score_loss
)
from imblearn.over_sampling import SMOTE

print("Core libraries imported successfully.")

## 2. Data Loading and Initial Inspection

In [ ]:
# Load dataset supporting multiple potential relative paths
data_paths = [
    "../creditcard.csv",
    "creditcard.csv",
    "../creditcard.csv.zip",
    "creditcard.csv.zip"
]

loaded_path = None
for p in data_paths:
    if os.path.exists(p):
        loaded_path = p
        break

try:
    credit_card_data = pd.read_csv(loaded_path if loaded_path else "creditcard.csv")
    print(f"Dataset successfully loaded from: {loaded_path}")
except Exception as e:
    print(f"Error loading data: {e}")
    raise e

print(f"Rows: {credit_card_data.shape[0]:,}")
print(f"Columns: {credit_card_data.shape[1]}")

In [ ]:
# Dataset info and missing values check
print("Dataset Info:")
print(credit_card_data.info())

missing_count = credit_card_data.isnull().sum().sum()
duplicate_count = credit_card_data.duplicated().sum()
print(f"Total Missing Values: {missing_count}")
print(f"Total Duplicate Rows: {duplicate_count}")

In [ ]:
# First and last 5 records
print("First 5 records:")
display(credit_card_data.head()) if 'display' in globals() else print(credit_card_data.head())

print("Last 5 records:")
display(credit_card_data.tail()) if 'display' in globals() else print(credit_card_data.tail())

## 3. Target Class and Feature Distributions

In [ ]:
# Class imbalance quantification
class_counts = credit_card_data['Class'].value_counts()
class_pcts = credit_card_data['Class'].value_counts(normalize=True) * 100

print("Class Distribution Summary:")
print(f"Legitimate (0): {class_counts[0]:,} ({class_pcts[0]:.3f}%)")
print(f"Fraudulent (1): {class_counts[1]:,} ({class_pcts[1]:.3f}%)")
print(f"Ratio: 1 fraud per {int(class_counts[0] / class_counts[1])} legitimate transactions")

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
sns.countplot(x='Class', data=credit_card_data, palette=['#2b5c8f', '#d9534f'], ax=ax[0])
ax[0].set_title('Transaction Class Counts', fontsize=12, fontweight='bold')
ax[0].set_xticklabels(['Normal (0)', 'Fraud (1)'])
for p in ax[0].patches:
    ax[0].annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height()),
                   ha='center', va='center', xytext=(0, 6), textcoords='offset points')

# Pie chart
ax[1].pie(class_counts, labels=['Normal', 'Fraud'], autopct='%1.2f%%',
          colors=['#2b5c8f', '#d9534f'], explode=[0, 0.2], shadow=True, startangle=45)
ax[1].set_title('Transaction Class Proportions', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Transaction Amount comparison
normal_amt = credit_card_data[credit_card_data['Class'] == 0]['Amount']
fraud_amt = credit_card_data[credit_card_data['Class'] == 1]['Amount']

print("Normal Transaction Amount Statistics:")
print(normal_amt.describe().round(2))
print("\nFraud Transaction Amount Statistics:")
print(fraud_amt.describe().round(2))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(normal_amt, bins=50, ax=ax[0], color='#2b5c8f', kde=True)
ax[0].set_title('Normal Transaction Amounts', fontweight='bold')
ax[0].set_yscale('log')
ax[0].set_xlabel('Amount')

sns.histplot(fraud_amt, bins=50, ax=ax[1], color='#d9534f', kde=True)
ax[1].set_title('Fraud Transaction Amounts', fontweight='bold')
ax[1].set_yscale('log')
ax[1].set_xlabel('Amount')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation analysis with Target Class
correlations = credit_card_data.corr()['Class'].sort_values()
top_neg = correlations.head(5)
top_pos = correlations.tail(6).drop('Class')

print("Top Negatively Correlated Features:")
print(top_neg.round(4))
print("\nTop Positively Correlated Features:")
print(top_pos.round(4))

top_feats = list(top_neg.index) + list(top_pos.index)
plt.figure(figsize=(10, 8))
sns.heatmap(credit_card_data[top_feats].corr(), annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation Heatmap of Key Discriminating Features', fontsize=12, fontweight='bold')
plt.show()

## 4. Feature Engineering and Stratified Splitting

In [ ]:
# Preprocessing: drop Time and log-transform Amount
df_proc = credit_card_data.copy()
df_proc.drop(columns=['Time'], inplace=True)
df_proc['LogAmount'] = np.log1p(df_proc['Amount'])
df_proc.drop(columns=['Amount'], inplace=True)

# Separate features and target
X = df_proc.drop(columns=['Class'])
y = df_proc['Class']

# Standardize features
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train samples: {X_train.shape[0]:,} (Frauds: {y_train.sum()})")
print(f"Test samples:  {X_test.shape[0]:,} (Frauds: {y_test.sum()})")

## 5. Model 1: Logistic Regression (Baseline vs Balanced)

In [ ]:
# Train unweighted baseline and class-weighted balanced model
lr_base = LogisticRegression(max_iter=1000, random_state=42)
lr_base.fit(X_train, y_train)

lr_bal = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_bal.fit(X_train, y_train)

pred_base = lr_base.predict(X_test)
pred_bal = lr_bal.predict(X_test)
prob_bal = lr_bal.predict_proba(X_test)[:, 1]

print("Baseline LR  - Recall: {:.2%}, Precision: {:.2%}, F1: {:.4f}".format(
    recall_score(y_test, pred_base), precision_score(y_test, pred_base), f1_score(y_test, pred_base)
))
print("Balanced LR  - Recall: {:.2%}, Precision: {:.2%}, F1: {:.4f}".format(
    recall_score(y_test, pred_bal), precision_score(y_test, pred_bal), f1_score(y_test, pred_bal)
))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(confusion_matrix(y_test, pred_base), annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False)
axes[0].set_title(f'Baseline LR (Recall: {recall_score(y_test, pred_base):.2%})')
axes[0].set_xticklabels(['Normal', 'Fraud'])
axes[0].set_yticklabels(['Normal', 'Fraud'])

sns.heatmap(confusion_matrix(y_test, pred_bal), annot=True, fmt='d', cmap='Greens', ax=axes[1], cbar=False)
axes[1].set_title(f'Balanced LR (Recall: {recall_score(y_test, pred_bal):.2%})')
axes[1].set_xticklabels(['Normal', 'Fraud'])
axes[1].set_yticklabels(['Normal', 'Fraud'])
plt.tight_layout()
plt.show()

## 6. Model 2: Decision Tree Classifier

In [ ]:
# Train constrained decision tree
dt = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=50,
    min_samples_leaf=20,
    class_weight='balanced',
    random_state=42
)
dt.fit(X_train, y_train)

pred_dt = dt.predict(X_test)
prob_dt = dt.predict_proba(X_test)[:, 1]

print("Decision Tree Results:")
print("Accuracy:  {:.4f}".format(accuracy_score(y_test, pred_dt)))
print("Precision: {:.4f}".format(precision_score(y_test, pred_dt)))
print("Recall:    {:.4f}".format(recall_score(y_test, pred_dt)))
print("F1-Score:  {:.4f}".format(f1_score(y_test, pred_dt)))
print("ROC-AUC:   {:.4f}".format(roc_auc_score(y_test, prob_dt)))

# Top feature importances
importances = pd.Series(dt.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
plt.figure(figsize=(9, 4))
sns.barplot(x=importances.values, y=importances.index, palette='viridis')
plt.title('Top 10 Feature Importances (Decision Tree)', fontsize=12, fontweight='bold')
plt.show()

## 7. Model 3: K-Nearest Neighbor (KNN)

In [ ]:
# Stratified subsample for fast KNN training
train_sub = pd.concat([X_train, y_train], axis=1)
f_sub = train_sub[train_sub['Class'] == 1]
n_sub = train_sub[train_sub['Class'] == 0].sample(n=len(f_sub) * 4, random_state=42)
knn_sample = pd.concat([f_sub, n_sub]).sample(frac=1, random_state=42)

X_train_knn = knn_sample.drop(columns=['Class'])
y_train_knn = knn_sample['Class']

knn = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)
knn.fit(X_train_knn, y_train_knn)

pred_knn = knn.predict(X_test)
prob_knn = knn.predict_proba(X_test)[:, 1]

print("KNN (k=5) Results:")
print("Precision: {:.4f}".format(precision_score(y_test, pred_knn)))
print("Recall:    {:.4f}".format(recall_score(y_test, pred_knn)))
print("F1-Score:  {:.4f}".format(f1_score(y_test, pred_knn)))
print("ROC-AUC:   {:.4f}".format(roc_auc_score(y_test, prob_knn)))

## 8. Model 4: Random Forest with SMOTE Oversampling

In [ ]:
# Apply SMOTE to training data only
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE - Normal: {(y_train == 0).sum():,}, Fraud: {(y_train == 1).sum():,}")
print(f"After SMOTE  - Normal: {(y_train_smote == 0).sum():,}, Fraud: {(y_train_smote == 1).sum():,}")

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train_smote, y_train_smote)

pred_rf = rf.predict(X_test)
prob_rf = rf.predict_proba(X_test)[:, 1]

print("Random Forest + SMOTE Results:")
print("Accuracy:  {:.4f}".format(accuracy_score(y_test, pred_rf)))
print("Precision: {:.4f}".format(precision_score(y_test, pred_rf)))
print("Recall:    {:.4f}".format(recall_score(y_test, pred_rf)))
print("F1-Score:  {:.4f}".format(f1_score(y_test, pred_rf)))
print("ROC-AUC:   {:.4f}".format(roc_auc_score(y_test, prob_rf)))

## 9. Unified Model Benchmark and Comparison

In [ ]:
# Compile evaluation table
benchmark_models = {
    'Logistic Regression (Balanced)': (pred_bal, prob_bal),
    'Decision Tree (Pruned)': (pred_dt, prob_dt),
    'K-Nearest Neighbors': (pred_knn, prob_knn),
    'Random Forest (+ SMOTE)': (pred_rf, prob_rf)
}

comparison_rows = []
for name, (preds, probs) in benchmark_models.items():
    comparison_rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds, zero_division=0),
        'Recall': recall_score(y_test, preds),
        'F1-Score': f1_score(y_test, preds, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, probs),
        'PR-AUC': average_precision_score(y_test, probs)
    })

comparison_df = pd.DataFrame(comparison_rows).set_index('Model')
print("Model Comparison Benchmark Table:")
display(comparison_df.round(4)) if 'display' in globals() else print(comparison_df.round(4))

In [ ]:
# Visual comparison: ROC and PR curves
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for name, (_, probs) in benchmark_models.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    prec, rec, _ = precision_recall_curve(y_test, probs)
    auc_v = roc_auc_score(y_test, probs)
    ap_v = average_precision_score(y_test, probs)
    
    axes[0].plot(fpr, tpr, lw=2, label=f'{name} (AUC={auc_v:.3f})')
    axes[1].plot(rec, prec, lw=2, label=f'{name} (AP={ap_v:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_title('Combined ROC Curves', fontsize=12, fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right')

axes[1].set_title('Combined Precision-Recall Curves', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

## 10. Financial Cost Optimization and Cross-Validation

In [ ]:
# Banking cost-utility curve across thresholds
# Average missed fraud loss: $122. Verification friction cost: $5.
thresholds = np.linspace(0.05, 0.95, 50)
cost_fn, cost_fp = 122.0, 5.0
losses = []

for t in thresholds:
    t_preds = (prob_bal >= t).astype(int)
    cm_t = confusion_matrix(y_test, t_preds)
    losses.append((cm_t[1, 0] * cost_fn) + (cm_t[0, 1] * cost_fp))

opt_idx = np.argmin(losses)
opt_thresh = thresholds[opt_idx]

plt.figure(figsize=(9, 4))
plt.plot(thresholds, losses, color='#d9534f', lw=2.5, label='Total Financial Loss ($)')
plt.axvline(x=opt_thresh, color='green', linestyle='--', label=f'Optimal Threshold ({opt_thresh:.2f})')
plt.title('Total Business Loss vs Fraud Decision Threshold', fontsize=12, fontweight='bold')
plt.xlabel('Threshold')
plt.ylabel('Loss ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Optimal Threshold: {opt_thresh:.2f}")
print(f"Minimum Expected Loss: ${losses[opt_idx]:,.2f}")

## 11. Production Inference and Real-Time Scoring (In-Memory)

In [ ]:
# Build end-to-end production pipeline in memory (no pkl files required)
X_raw = credit_card_data.drop(columns=['Class'])
y_raw = credit_card_data['Class']

X_tr_raw, X_te_raw, y_tr_raw, y_te_raw = train_test_split(
    X_raw, y_raw, test_size=0.2, stratify=y_raw, random_state=42
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

print("Fitting in-memory production pipeline...")
pipeline.fit(X_tr_raw, y_tr_raw)
print("Pipeline ready for real-time transactions.")

In [ ]:
def predict_transaction(tx_record, threshold=0.30):
    '''
    Evaluates incoming transaction record in real time.
    '''
    df_in = pd.DataFrame([tx_record])
    for col in X_raw.columns:
        if col not in df_in.columns:
            df_in[col] = 0.0
    df_in = df_in[X_raw.columns]
    
    prob = pipeline.predict_proba(df_in)[0, 1]
    is_fraud = bool(prob >= threshold)
    
    if prob < 0.20:
        risk = "LOW RISK - APPROVE"
    elif prob < 0.50:
        risk = "MEDIUM RISK - 2FA CHALLENGE"
    else:
        risk = "CRITICAL RISK - BLOCK TRANSACTION"
        
    return {
        "is_fraud": is_fraud,
        "fraud_probability": round(float(prob), 4),
        "risk_level": risk,
        "action": "DECLINE" if is_fraud else "APPROVE"
    }

# Live tests
legit_sample = X_te_raw[y_te_raw == 0].iloc[10].to_dict()
fraud_sample = X_te_raw[y_te_raw == 1].iloc[5].to_dict()

print("Test Case 1 - Legitimate Transaction:")
print(predict_transaction(legit_sample))

print("\nTest Case 2 - Fraudulent Transaction:")
print(predict_transaction(fraud_sample))

In [ ]:
# Latency and throughput benchmark
iterations = 1000
start = time.perf_counter()
for _ in range(iterations):
    _ = predict_transaction(legit_sample)
duration = time.perf_counter() - start

latency_ms = (duration / iterations) * 1000
throughput = iterations / duration

print(f"Average Latency: {latency_ms:.3f} ms per transaction")
print(f"Throughput:      {throughput:,.0f} transactions per second")
print("Full end-to-end pipeline execution complete.")